In [2]:
from pathlib import Path

# Configuration
CATEGORIES = ["bottle", "hazelnut", "transistor"]

# Chemin utilisé dans l'US-02-01
DATASET_ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

# Vérification de la racine
assert DATASET_ROOT.is_dir(), (
    f"Dataset introuvable : {DATASET_ROOT}"
)

print("Dataset :", DATASET_ROOT)

# Vérifier les catégories et leurs images normales
for category in CATEGORIES:
    train_dir = DATASET_ROOT / category / "train" / "good"
    test_dir = DATASET_ROOT / category / "test"

    assert train_dir.is_dir(), f"Train manquant : {category}"
    assert test_dir.is_dir(), f"Test manquant : {category}"

    images = sorted(train_dir.glob("*.png"))

    print(f"{category}: {len(images)} images normales")

Dataset : /kaggle/input/datasets/ipythonx/mvtec-ad
bottle: 209 images normales
hazelnut: 391 images normales
transistor: 213 images normales


In [6]:
import random

# Configuration du protocole
SEED = 42
VALIDATION_RATIO = 0.20

# Stockage des splits pour les trois catégories
splits = {}

for category in CATEGORIES:

    # 1. Récupérer uniquement les images normales du train officiel
    train_dir = DATASET_ROOT / category / "train" / "good"

    images = sorted(
        path.relative_to(DATASET_ROOT).as_posix()
        for path in train_dir.glob("*.png")
    )

    assert images, f"Aucune image trouvée : {category}"

    # 2. Mélanger de manière reproductible
    shuffled = images.copy()
    rng = random.Random(SEED)
    rng.shuffle(shuffled)

    # 3. Calculer le nombre d'images de validation
    n_val = round(len(shuffled) * VALIDATION_RATIO)

    # 4. Construire les deux ensembles
    val_images = sorted(shuffled[:n_val])
    train_images = sorted(shuffled[n_val:])

    # 5. Conserver les résultats
    splits[category] = {
        "train": train_images,
        "val": val_images,
    }

    # 6. Vérifications de base
    assert len(train_images) + len(val_images) == len(images)
    assert not (set(train_images) & set(val_images))
    assert set(train_images) | set(val_images) == set(images)

    assert all(
        p.startswith(f"{category}/train/good/")
        for p in train_images + val_images
    )

    print(f"=== {category} ===")
    print(f"Total      : {len(images)}")
    print(f"Train      : {len(train_images)}")
    print(f"Validation : {len(val_images)}")
    print()

=== bottle ===
Total      : 209
Train      : 167
Validation : 42

=== hazelnut ===
Total      : 391
Train      : 313
Validation : 78

=== transistor ===
Total      : 213
Train      : 170
Validation : 43



In [7]:
import csv
from pathlib import Path

# Répertoire de sortie Kaggle
OUTPUT_DIR = Path("/kaggle/working/mvtec_splits")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "category",
    "split",
    "image_path",
    "label",
    "defect_type",
    "mask_path",
]

def relative_path(path):
    """Chemin relatif à la racine MVTec AD."""
    return path.relative_to(DATASET_ROOT).as_posix()


for category in CATEGORIES:
    rows = []

    # 1. Images normales : Train et Validation
    for split in ["train", "val"]:
        for image_path in splits[category][split]:
            rows.append({
                "category": category,
                "split": split,
                "image_path": image_path,
                "label": 0,
                "defect_type": "good",
                "mask_path": "",
            })

    # 2. Test officiel : images normales et anormales
    test_dir = DATASET_ROOT / category / "test"

    for defect_dir in sorted(test_dir.iterdir()):
        if not defect_dir.is_dir():
            continue

        defect_type = defect_dir.name
        is_anomaly = defect_type != "good"

        for image_file in sorted(defect_dir.glob("*.png")):
            mask_path = ""

            if is_anomaly:
                mask_file = (
                    DATASET_ROOT
                    / category
                    / "ground_truth"
                    / defect_type
                    / f"{image_file.stem}_mask.png"
                )

                assert mask_file.is_file(), (
                    f"Masque manquant : {mask_file}"
                )

                mask_path = relative_path(mask_file)

            rows.append({
                "category": category,
                "split": "test",
                "image_path": relative_path(image_file),
                "label": int(is_anomaly),
                "defect_type": defect_type,
                "mask_path": mask_path,
            })

    # 3. Export CSV
    manifest_path = OUTPUT_DIR / f"{category}_manifest.csv"

    with manifest_path.open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=COLUMNS,
            lineterminator="\n",
        )
        writer.writeheader()
        writer.writerows(rows)

    # 4. Afficher un résumé
    counts = {
        split: sum(row["split"] == split for row in rows)
        for split in ["train", "val", "test"]
    }

    print(f"=== {category} ===")
    print(f"Manifest : {manifest_path.name}")
    print(f"Train    : {counts['train']}")
    print(f"Val      : {counts['val']}")
    print(f"Test     : {counts['test']}")
    print(f"Total    : {len(rows)}")
    print()

=== bottle ===
Manifest : bottle_manifest.csv
Train    : 167
Val      : 42
Test     : 83
Total    : 292

=== hazelnut ===
Manifest : hazelnut_manifest.csv
Train    : 313
Val      : 78
Test     : 110
Total    : 501

=== transistor ===
Manifest : transistor_manifest.csv
Train    : 170
Val      : 43
Test     : 100
Total    : 313



In [5]:
import csv
import hashlib

total_images = 0
baseline_hashes = {}

for category in CATEGORIES:

    manifest_path = OUTPUT_DIR / f"{category}_manifest.csv"

    # Lire le manifeste
    with manifest_path.open(encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        assert reader.fieldnames == COLUMNS
        rows = list(reader)

    # 1. Extraire les chemins par split
    train = {
        r["image_path"] for r in rows if r["split"] == "train"
    }
    val = {
        r["image_path"] for r in rows if r["split"] == "val"
    }
    test = {
        r["image_path"] for r in rows if r["split"] == "test"
    }

    # 2. Vérifier les doublons et les chevauchements
    all_paths = [r["image_path"] for r in rows]

    assert len(all_paths) == len(set(all_paths))
    assert train.isdisjoint(val)
    assert train.isdisjoint(test)
    assert val.isdisjoint(test)

    # 3. Comparer avec les images originales
    original_train = {
        p.relative_to(DATASET_ROOT).as_posix()
        for p in (DATASET_ROOT / category / "train" / "good").glob("*.png")
    }

    original_test = {
        p.relative_to(DATASET_ROOT).as_posix()
        for p in (DATASET_ROOT / category / "test").glob("*/*.png")
    }

    assert train | val == original_train
    assert test == original_test

    # 4. Vérifier les labels et les masques
    for row in rows:
        assert row["category"] == category
        assert row["split"] in {"train", "val", "test"}

        image_path = row["image_path"]
        assert (DATASET_ROOT / image_path).is_file()

        if row["split"] in {"train", "val"}:
            assert row["label"] == "0"
            assert row["defect_type"] == "good"
            assert row["mask_path"] == ""

        else:
            defect_type = row["defect_type"]

            assert image_path.startswith(
                f"{category}/test/{defect_type}/"
            )

            if defect_type == "good":
                assert row["label"] == "0"
                assert row["mask_path"] == ""
            else:
                assert row["label"] == "1"

                expected_mask = (
                    f"{category}/ground_truth/{defect_type}/"
                    f"{Path(image_path).stem}_mask.png"
                )
                assert row["mask_path"] == expected_mask
                assert (DATASET_ROOT / expected_mask).is_file()

    # 5. Calculer l'empreinte SHA-256
    sha256 = hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest()

    baseline_hashes[category] = sha256
    total_images += len(rows)

    print(f"=== {category} ===")
    print("Doublons                 : 0")
    print("Chevauchement            : 0")
    print("Train + Val complets     : OK")
    print("Test officiel préservé   : OK")
    print("Labels et masques        : OK")
    print(f"SHA-256                  : {sha256}")
    print()

assert total_images == 1106

print("TOTAL :", total_images, "images")
print("TOUTES LES VÉRIFICATIONS : OK")

=== bottle ===
Doublons                 : 0
Chevauchement            : 0
Train + Val complets     : OK
Test officiel préservé   : OK
Labels et masques        : OK
SHA-256                  : 32b3c0998b3ebb1305aef61d7153bb80dc14b9c9ab4975241435dd7c8b322064

=== hazelnut ===
Doublons                 : 0
Chevauchement            : 0
Train + Val complets     : OK
Test officiel préservé   : OK
Labels et masques        : OK
SHA-256                  : 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6

=== transistor ===
Doublons                 : 0
Chevauchement            : 0
Train + Val complets     : OK
Test officiel préservé   : OK
Labels et masques        : OK
SHA-256                  : eff9f514691610284c902020e62931189ffae80d20fabff1f4a6ab702d8ef728

TOTAL : 1106 images
TOUTES LES VÉRIFICATIONS : OK


In [8]:
import hashlib

print("=== Vérification de reproductibilité ===\n")

for category in CATEGORIES:
    manifest_path = OUTPUT_DIR / f"{category}_manifest.csv"

    # Empreinte enregistrée lors de la première génération
    original_hash = baseline_hashes[category]

    # Empreinte du fichier régénéré
    new_hash = hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest()

    # Comparaison
    identical = original_hash == new_hash

    print(f"=== {category} ===")
    print(f"Original : {original_hash}")
    print(f"Nouveau  : {new_hash}")
    print(f"Identique : {identical}")
    print()

    assert identical, (
        f"Échec de reproductibilité pour {category}"
    )

print("REPRODUCTIBILITÉ VALIDÉE POUR LES 3 CATÉGORIES")

=== Vérification de reproductibilité ===

=== bottle ===
Original : 32b3c0998b3ebb1305aef61d7153bb80dc14b9c9ab4975241435dd7c8b322064
Nouveau  : 32b3c0998b3ebb1305aef61d7153bb80dc14b9c9ab4975241435dd7c8b322064
Identique : True

=== hazelnut ===
Original : 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6
Nouveau  : 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6
Identique : True

=== transistor ===
Original : eff9f514691610284c902020e62931189ffae80d20fabff1f4a6ab702d8ef728
Nouveau  : eff9f514691610284c902020e62931189ffae80d20fabff1f4a6ab702d8ef728
Identique : True

REPRODUCTIBILITÉ VALIDÉE POUR LES 3 CATÉGORIES


In [9]:
import json
import hashlib
import zipfile

# Empreintes vérifiées lors de notre expérience
EXPECTED_HASHES = {
    "bottle": "32b3c0998b3ebb1305aef61d7153bb80dc14b9c9ab4975241435dd7c8b322064",
    "hazelnut": "8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6",
    "transistor": "eff9f514691610284c902020e62931189ffae80d20fabff1f4a6ab702d8ef728",
}

protocol = {
    "dataset": "MVTec AD",
    "dataset_source": "ipythonx/mvtec-ad",
    "seed": SEED,
    "validation_ratio": VALIDATION_RATIO,
    "categories": CATEGORIES,
    "split_method": (
        "sort relative paths, random.Random(seed).shuffle, "
        "round(n * validation_ratio), validation first"
    ),
    "train_validation_source": "train/good",
    "test_source": "test",
    "test_policy": "Reserved for final evaluation only",
    "manifest_columns": COLUMNS,
    "manifests": {},
}

# Vérifier et enregistrer les empreintes
for category in CATEGORIES:
    path = OUTPUT_DIR / f"{category}_manifest.csv"

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    assert digest == EXPECTED_HASHES[category], (
        f"SHA-256 différent pour {category}"
    )

    protocol["manifests"][category] = {
        "filename": path.name,
        "sha256": digest,
    }

# Enregistrer la configuration
config_path = OUTPUT_DIR / "split_protocol.json"

config_path.write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

# Créer une archive contenant les quatre fichiers
archive_path = Path("/kaggle/working/us-02-03-artifacts.zip")

with zipfile.ZipFile(
    archive_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(OUTPUT_DIR.glob("*")):
        archive.write(path, arcname=path.name)

print("Archive créée :", archive_path)
print("\nContenu :")

with zipfile.ZipFile(archive_path) as archive:
    for filename in archive.namelist():
        print(" -", filename)

print("\nEXPORT PRÊT")

Archive créée : /kaggle/working/us-02-03-artifacts.zip

Contenu :
 - bottle_manifest.csv
 - hazelnut_manifest.csv
 - split_protocol.json
 - transistor_manifest.csv

EXPORT PRÊT
